In [ ]:
#Kuramoto Sivashinsky equation comparisons for 41 dimensions(20 Nodes) 
import numpy as np
import sympy as sp

# Set random seeds for reproducibility
np.random.seed(1)

# Parameters
N = 20  # Number of Fourier modes (so dim_x = 2N + 1)
dim_x = 2 * N + 1
dim_y = N  # Number of measurements
L = 2*np.pi
c, d = -L, L
dt = 1e-6
seq_length = 3001
num_records = 50

# Physical parameter
nu = 0.01

# Linear eigenvalues
lambda_array = [(i ** 2) - (nu * (i ** 4)) for i in range(1, N + 1)]

# Measurement matrix H
m_points = c + (d - c) * np.random.random_sample(dim_y)
H = np.zeros((dim_y, dim_x))
for i in range(dim_y):
    H[i, 0] = 1
    for j in range(N):
        H[i, j + 1] = np.cos((j + 1) * 2 * np.pi * m_points[i] / L)
        H[i, j + N + 1] = np.sin((j + 1) * 2 * np.pi * m_points[i] / L)

# Noise parameters
c_p = 0.0001
c_q = 0.0001
c_r = 100 ########## 10 1 0.1 0.01 0.001 0.0001 ########## 1 time per gpu (for other files)
P = c_p * np.eye(dim_x)
Q = c_q * np.eye(dim_x)
R = c_r * np.eye(dim_y)

data_initial_state_mean=np.load("x0_mean.npz")
initial_state_mean=data_initial_state_mean["x0_mean"]

def ks_galerkin_rhs(x, N, nu):
    """
    Compute RHS of 21D Galerkin approximation of the KS equation.
    x: state vector [a0, a1, ..., aN, b1, ..., bN] of length 2N + 1
    Returns dx/dt vector of same shape
    """
    assert len(x) == 2 * N + 1
    a0 = x[0]
    a = x[1:N+1]
    b = x[N+1:]

    # Preallocate derivative
    dxdt = np.zeros_like(x)
    dxdt[0] = 0  # a0 is constant due to zero mean in KS

    # Linear part
    for k in range(1, N+1):
        L_k = -(nu * (k**2) + (k**4))
        dxdt[k] = L_k * a[k-1]
        dxdt[N + k] = L_k * b[k-1]

    # Nonlinear part (convolution sum)
    for n in range(1, N+1):
        sum_cos = 0
        sum_sin = 0
        for k in range(1, N+1):
            for m in range(1, N+1):
                if abs(k - m) == n:
                    sum_cos += 0.5 * (a[k-1]*a[m-1] + b[k-1]*b[m-1])
                    sum_sin += a[k-1]*b[m-1]
                if (k + m) == n:
                    sum_cos += 0.5 * (a[k-1]*a[m-1] - b[k-1]*b[m-1])
                    sum_sin += a[k-1]*b[m-1]

        dxdt[n] += -n * sum_sin
        dxdt[N + n] += n * sum_cos

    return dxdt

# Euler integrator
def step_forward(x, N, nu):
    return x + dt * ks_galerkin_rhs(x, N, nu)

# Generate data
X_data_array = np.empty((num_records, seq_length, dim_x))
Y_data_array = np.empty((num_records, seq_length, dim_y))

#initial condition
z = sp.symbols('z')
u0 = 5*z - 0.5 * z**2 - 4

# Orthonormal basis for calculating Fourier coefficients
phi = [1/sp.sqrt(2*sp.pi)]  # for a0
psi = []

for i in range(1, N+1):  # for a1, ..., aN and b1, ..., bN #check this
    phi.append((1/sp.sqrt(sp.pi)) * sp.cos(z*i))
    psi.append((1/sp.sqrt(sp.pi)) * sp.sin(z*i))

for i in range(num_records):
    # perturbed_mean = x0_mean + np.random.uniform(-2, 2, size=dim_x)
    perturbed_mean=np.random.uniform(x0_mean-0.5,x0_mean+0.5,size=dim_x)
    x = np.random.multivariate_normal(perturbed_mean, P)

    X_data_array[i, 0] = x
    Y_data_array[i, 0] = H @ x + np.random.multivariate_normal(np.zeros(dim_y), R)

    for j in range(1, seq_length):
        w = np.random.multivariate_normal(np.zeros(dim_x), Q)
        x = step_forward(x, N, nu) + w
        X_data_array[i, j] = x
        Y_data_array[i, j] = H @ x + np.random.multivariate_normal(np.zeros(dim_y), R)

np.savez(f"ks_data_cr_{str(c_r).replace('.', '_')}.npz", X_data=X_data_array, Y_data=Y_data_array)
np.savez('matrix_h.npz',H=H)
np.savez('x0_mean.npz',x0_mean=x0_mean)
print(X_data_array[0])

[-1.41497513e+01  3.54490770e+00 -8.86226925e-01  3.93878634e-01
 -2.21556731e-01  1.41796308e-01 -9.84696584e-02  7.23450551e-02
 -5.53891828e-02  4.37642926e-02 -3.54490770e-02  2.92967579e-02
 -2.46174146e-02  2.09757852e-02 -1.80862638e-02  1.57551453e-02
 -1.38472957e-02  1.22661166e-02 -1.09410732e-02  9.81968892e-03
 -8.86226925e-03  1.77245385e+01 -8.86226925e+00  5.90817950e+00
 -4.43113463e+00  3.54490770e+00 -2.95408975e+00  2.53207693e+00
 -2.21556731e+00  1.96939317e+00 -1.77245385e+00  1.61132168e+00
 -1.47704488e+00  1.36342604e+00 -1.26603846e+00  1.18163590e+00
 -1.10778366e+00  1.04261991e+00 -9.84696584e-01  9.32870448e-01
 -8.86226925e-01]
[[-1.38320885e+01  4.00568974e+00 -1.07861072e+00 ... -8.95906346e-01
   1.13276010e+00 -1.28187011e+00]
 [-1.38160134e+01  4.00338212e+00 -1.08224802e+00 ... -7.96801486e-01
   9.89578725e-01 -1.05573547e+00]
 [-1.38243490e+01  3.99966240e+00 -1.08791231e+00 ... -7.05926572e-01
   8.65285146e-01 -8.76132842e-01]
 ...
 [-1.2885363

In [4]:
# Generate data
X_data_array_diff_ic = np.empty((num_records, seq_length, dim_x))
Y_data_array_diff_ic = np.empty((num_records, seq_length, dim_y))

for i in range(num_records):
    perturbed_mean_2=np.random.uniform(x0_mean-0.8,x0_mean+0.8,size=dim_x)
    x_2 = np.random.multivariate_normal(perturbed_mean_2, P)
    X_data_array_diff_ic[i, 0] = x_2
    Y_data_array_diff_ic[i, 0] = H @ x_2 + np.random.multivariate_normal(np.zeros(dim_y), R)

    for j in range(1, seq_length):
        w = np.random.multivariate_normal(np.zeros(dim_x), Q)
        x_2 = step_forward(x_2, N, nu) + w
        X_data_array_diff_ic[i, j] = x_2
        Y_data_array_diff_ic[i, j] = H @ x_2 + np.random.multivariate_normal(np.zeros(dim_y), R)
        
np.savez(f"ks_data_diff_ic_cr_{str(c_r).replace('.', '_')}.npz", X_data=X_data_array_diff_ic, Y_data=Y_data_array_diff_ic)